# 5주차 수업 정리: 퍼셉트론과 딥러닝 모델 설계

5주차 수업 내용과 교수님이 올려주신 **딥러닝 모델 설계 시 주의사항**을 정리했습니다.
실습 노트북과 함께 보면 좋습니다.

- `01_Perceptron_AND.ipynb`: 퍼셉트론 하나로 AND / OR 게이트 계산
- `02_Wine_DL.ipynb`: diabetes 예제를 Wine 다중분류로 바꿔 적용

1. 딥러닝을 이루는 5가지 요소
2. 밸브 조정 = 최적화 문제
3. 활성화 함수가 필요한 이유 (비선형성)
4. 딥러닝 vs 머신러닝: 데이터 준비의 차이
5. 모델 설계 시 주의사항 (교수님 자료)
6. 문제 종류별 마지막 층과 loss 선택표

In [1]:
import numpy as np

## 1. 딥러닝을 이루는 5가지 요소

| 요소 | 하는 일 | Wine 실습에서는 |
|---|---|---|
| Forward Propagation | 입력 → 가중치 곱 + 바이어스 → 활성화 함수 → 다음 층 … → 예측값 | 13개 특성 → 32 → 16 → 3개 확률 |
| Loss function | 예측값이 정답과 얼마나 다른지 숫자로 계산 | `categorical_crossentropy` |
| Backward Propagation | loss를 줄이려면 각 가중치를 어느 방향으로 바꿔야 하는지(기울기) 계산 | `model.fit` 안에서 자동 |
| Optimizer (SGD 등) | 계산된 기울기를 보고 가중치를 실제로 조정 | `adam` (SGD를 발전시킨 방법) |
| Activation function | 층과 층 사이에 비선형성을 줌 | `relu`, 마지막 층 `softmax` |

한 번의 학습은 **Forward → Loss 계산 → Backward → Optimizer로 가중치 수정**의 반복입니다.
`epochs=100`이면 이 과정을 전체 데이터에 대해 100번 반복합니다.

퍼셉트론 용어로 보면

- 가중치: 입력이 있을 때 어떤 입력을 더 중요하게 여길 것인가
- 바이어스: 출력이 나오는 기준(문턱)을 조정하는 편향
- 기울기(가중치)는 데이터를 분류(회귀)하기 위한 기준선을 만드는 값

## 2. 밸브 조정 = 최적화 문제

수업에서는 가중치와 바이어스를 **밸브**에 비유했습니다. 예시 신경망에 밸브가 28개 있다면,
"28개 밸브의 값을 어떻게 조정해야 loss가 가장 작아지는가?"가 바로 **최적화 문제**입니다.

딥러닝이란 결국 **데이터의 특성을 파악해서 예측(분류/회귀)하는 것**이고,
그 방법이 밸브(파라미터)를 데이터에 맞게 자동으로 조정하는 것입니다.

밸브 개수는 층마다 `입력 수 × 뉴런 수 + 뉴런 수(바이어스)`로 계산합니다. Wine 모델로 직접 세어 보겠습니다.

In [2]:
layers = [13, 32, 16, 3]   # 입력 13 → 32 → 16 → 출력 3

total = 0
for n_in, n_out in zip(layers[:-1], layers[1:]):
    params = n_in * n_out + n_out   # 가중치 + 바이어스
    total += params
    print(f"{n_in:>2} → {n_out:>2} : {n_in} × {n_out} + {n_out} = {params}")

print("전체 밸브 수:", total)   # model.summary()의 Total params와 같음

13 → 32 : 13 × 32 + 32 = 448
32 → 16 : 32 × 16 + 16 = 528
16 →  3 : 16 × 3 + 3 = 51
전체 밸브 수: 1027


## 3. 활성화 함수가 필요한 이유 (비선형성)

수업 내용: **뉴런을 아무리 많이 쌓더라도 (활성화 함수가 없으면) 더 좋은 특성을 추출하기 어렵다.**

활성화 함수 없이 층을 쌓으면 `W2 · (W1 · x)`인데, 이는 `(W2 · W1) · x`와 같습니다.
즉 **여러 층을 쌓아도 직선(선형) 하나와 똑같아집니다.**
그래서 층과 층 사이에 ReLU 같은 활성화 함수로 **비선형성을 부과**해야 층을 쌓는 의미가 생깁니다.

In [3]:
np.random.seed(2)
x = np.random.randn(3)        # 입력 3개
W1 = np.random.randn(4, 3)    # 1층 가중치
W2 = np.random.randn(2, 4)    # 2층 가중치

two_layers = W2 @ (W1 @ x)    # 활성화 함수 없이 2층
one_layer = (W2 @ W1) @ x     # 가중치를 미리 곱한 1층

print("2층 결과:", two_layers.round(4))
print("1층 결과:", one_layer.round(4))
print("같은가?", np.allclose(two_layers, one_layer))

2층 결과: [-5.2829 -1.4232]
1층 결과: [-5.2829 -1.4232]
같은가? True


In [4]:
relu = lambda z: np.maximum(0, z)

print("1층 출력(W1 · x):", (W1 @ x).round(4))
print("ReLU 통과 후     :", relu(W1 @ x).round(4))   # 음수가 0으로 바뀜

with_relu = W2 @ relu(W1 @ x)  # 층 사이에 ReLU

print("ReLU 넣은 2층:", with_relu.round(4))
print("1층과 같은가?", np.allclose(with_relu, one_layer))

1층 출력(W1 · x): [ 1.2155  2.1205 -4.5488 -1.1059]
ReLU 통과 후     : [1.2155 2.1205 0.     0.    ]
ReLU 넣은 2층: [-0.7652 -1.851 ]
1층과 같은가? False


## 4. 딥러닝 vs 머신러닝: 데이터 준비의 차이

scikit-learn 머신러닝 모델은 DataFrame과 정수 라벨을 그대로 받아도 되지만, 딥러닝(Keras)에서는 다음을 챙깁니다.

1. **X값은 NumPy 배열로 변경**: `df.drop(...).values`
2. **y값은 원-핫 인코딩(One-hot Encoding) 형태로 변경**: 다중분류일 때

원-핫 인코딩은 클래스 번호를 "해당 위치만 1인 벡터"로 바꾸는 것입니다.

| 정수 라벨 | 원-핫 |
|---|---|
| 0 | [1, 0, 0] |
| 1 | [0, 1, 0] |
| 2 | [0, 0, 1] |

출력층 softmax가 내는 `[0.1, 0.8, 0.1]` 같은 확률 벡터와 모양을 맞추기 위해서입니다.

In [5]:
from sklearn.datasets import load_wine
from tensorflow.keras.utils import to_categorical

wine = load_wine(as_frame=True)
X = wine.frame.drop("target", axis=1).values   # 1. X → NumPy
y = wine.frame["target"].values

y_onehot = to_categorical(y)                    # 2. y → 원-핫

print("X 타입:", type(X), X.shape)
print("y 앞 3개 (정수):", y[[0, 60, 140]])
print("y 앞 3개 (원-핫):")
print(y_onehot[[0, 60, 140]])

X 타입: <class 'numpy.ndarray'> (178, 13)
y 앞 3개 (정수): [0 1 2]
y 앞 3개 (원-핫):
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


원-핫 인코딩을 사용하는 이유

- 출력층 softmax는 클래스마다 확률을 하나씩 냅니다. 정답도 같은 모양이어야 클래스별로 비교해서 loss를 계산할 수 있습니다.
- 정수 0, 1, 2를 그대로 쓰면 "2가 1보다 크다"는 크기·순서 관계가 있는 것처럼 보이지만, 클래스 사이에는 그런 관계가 없습니다. 원-핫은 모든 클래스를 동등하게 표현합니다.

원-핫 정답에는 `categorical_crossentropy`를 짝지어 사용합니다.
참고로 Keras에는 정수 라벨을 그대로 받는 `sparse_categorical_crossentropy`도 있습니다.

| y 형태 | loss |
|---|---|
| 원-핫 `[0, 1, 0]` | `categorical_crossentropy` |
| 정수 `1` | `sparse_categorical_crossentropy` |

y 형태와 loss가 **짝이 맞지 않으면** 오류가 나거나 학습이 이상해지므로 주의합니다.

## 5. 모델 설계 시 주의사항 (교수님 자료)

diabetes 예제에서 표시된 부분은 **데이터가 바뀌면 반드시 같이 바꿔야 하는 곳**입니다.

```python
model.add(tf.keras.layers.Dense(32, activation="relu", input_shape=(8,)))   # ① Feature number
model.add(tf.keras.layers.Dense(16, activation="relu"))
model.add(tf.keras.layers.Dense(1, activation="sigmoid"))                   # ② 출력 뉴런 수, ③ 마지막 활성화 함수

model.compile(optimizer="adam",
              loss="binary_crossentropy",                                   # ④ loss
              metrics=["accuracy"])                                         # ⑤ 평가 지표
```

| 번호 | 위치 | 결정 기준 | diabetes | Wine |
|---|---|---|---|---|
| ① | `input_shape` | 입력 특성(열) 개수 | 8 | 13 |
| ② | 마지막 `Dense` 뉴런 수 | 문제 종류 / 클래스 수 | 1 | 3 |
| ③ | 마지막 `activation` | 문제 종류 | sigmoid | softmax |
| ④ | `loss` | 문제 종류 (+ y 형태) | binary_crossentropy | categorical_crossentropy |
| ⑤ | `metrics` | 문제 종류 | accuracy | accuracy |

②~⑤는 **이진분류 / 다중분류 / 회귀 여부에 따라 각각 다릅니다.**
중간 은닉층(32, 16, relu)은 문제 종류와 상관없이 자유롭게 정할 수 있는 부분입니다.

추가로 확인할 점:

- 예제에는 `model.compile`이 5번과 6번에 두 번 있습니다. 한 번만 하면 되고, 두 번 하면 마지막 설정이 적용됩니다.
- 예측 후처리도 문제 종류에 따라 다릅니다. 이진분류는 `>= 0.5`, 다중분류는 `argmax`를 씁니다.

## 6. 문제 종류별 마지막 층과 loss 선택표

| 문제 종류 | 예시 | 마지막 층 | loss | metrics | 예측값 처리 |
|---|---|---|---|---|---|
| 이진분류 | diabetes (0/1) | `Dense(1, activation="sigmoid")` | `binary_crossentropy` | `accuracy` | `(y_prob >= 0.5)` |
| 다중분류 | Wine (0/1/2) | `Dense(클래스 수, activation="softmax")` | `categorical_crossentropy` (원-핫) / `sparse_categorical_crossentropy` (정수) | `accuracy` | `argmax` |
| 회귀 | 집값, 온도 | `Dense(1)` (활성화 함수 없음) | `mse` | `mae` | 그대로 사용 |

외우는 방법:

- **sigmoid**: 0~1 사이 확률 **하나** → "1일 확률"
- **softmax**: 클래스별 확률 **여러 개**, 합이 1 → "각 클래스일 확률"
- **회귀**: 값의 범위를 제한하면 안 되므로 활성화 함수 없이 그대로 출력